In [1]:
import os
from dotenv import load_dotenv
import googleapiclient.discovery


def get_youtube_client(env_var: str = "YOUTUBE_API_KEY"):

    load_dotenv()
    api_key = os.getenv(env_var)
    return googleapiclient.discovery.build(
        serviceName="youtube",
        version="v3",
        developerKey=api_key,
    )

youtube = get_youtube_client()

In [2]:
import pandas as pd

def get_channel_playlist_id(youtube, handle, kind: str = "uploads") -> str:
    prefix_map = {
        "uploads": "UU",
        "shorts": "UUSH",
        "videos": "UULF",
    } # https://stackoverflow.com/a/76602819
    if kind not in prefix_map:
        raise ValueError(f"Unsupported kind '{kind}'. Choose from {list(prefix_map)}.")

    resp = youtube.channels().list(part="id", forHandle=handle).execute()
    items = resp.get("items", [])
    if not items:
        raise ValueError(f"No channel found for handle: {handle}")

    channel_id = items[0]["id"]
    if not channel_id.startswith("UC"):
        raise ValueError(f"Expected channel ID starting with 'UC', got: {channel_id}")

    return prefix_map[kind] + channel_id[2:]

def get_playlist_video_ids(youtube, playlist_id, max_videos) -> list[str]:
    video_ids: list[str] = []
    page_token = None

    while len(video_ids) < max_videos:
        remaining = max_videos - len(video_ids)
        resp = (
            youtube.playlistItems().list(
                part="contentDetails",
                playlistId=playlist_id,
                maxResults=min(50, remaining),
                pageToken=page_token,
            ).execute()
        )
        for item in resp.get("items", []):
            vid = item.get("contentDetails", {}).get("videoId")
            if vid:
                video_ids.append(vid)
                if len(video_ids) >= max_videos:
                    break

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    return video_ids

def get_videos_df(youtube, video_ids, batch_size: int = 50) -> pd.DataFrame:
    rows = []

    for i in range(0, len(video_ids), batch_size):
        batch = video_ids[i:i+batch_size]
        resp = (
            youtube.videos().list(
                part="snippet,statistics",
                id=",".join(batch),
                maxResults=batch_size,
            )
            .execute()
        )

        for item in resp.get("items", []):
            snippet = item.get("snippet", {})
            stats = item.get("statistics", {})
            raw_views = stats.get("viewCount")
            video_id = item["id"]
            rows.append(
                {
                    "video_id": video_id,
                    "title": snippet.get("title"),
                    "published_at": snippet.get("publishedAt"),
                    "view_count": int(raw_views) if raw_views is not None else None,
                    "url": f"https://www.youtube.com/watch?v={video_id}"
                }
            )
    df = pd.DataFrame(rows)
    return df

In [3]:
handle = "@NFL" # CHANGE TO WHATEVER HANDLE NEEDED

videos_playlist_id = get_channel_playlist_id(youtube, handle="@NFL", kind="videos")
shorts_playlist_id = get_channel_playlist_id(youtube, handle="@NFL", kind="shorts")

videos_ids = get_playlist_video_ids(youtube, videos_playlist_id, max_videos=10000)
shorts_ids = get_playlist_video_ids(youtube, shorts_playlist_id, max_videos=10000)

videos_df = get_videos_df(youtube, videos_ids)
shorts_df = get_videos_df(youtube, shorts_ids)

In [ ]:
# videos_df.to_csv("data/nfl_videos.csv", index=False) # change csv titles as appropriate
# shorts_df.to_csv("data/nfl_shorts.csv", index=False)